# Module 1: Setup and Base Model Evaluation

**Duration:** ~20 minutes

**Environment:** AWS g4dn.4xlarge (or local GPU)

## Learning Objectives
- Verify GPU environment is properly configured
- Load TinyLlama-1.1B with 4-bit quantization
- Evaluate baseline model performance on F5 technical questions

## Overview
In this module, we'll set up our environment and establish a baseline for how a general-purpose LLM performs on F5-specific technical questions. This baseline will be compared against RAG-enhanced and fine-tuned models in later modules.

## Prerequisites
- Virtual environment activated with dependencies installed
- Run from the `llm-finetuning-rag-lab` directory

## 1.1 Environment Check

Let's verify GPU access and CUDA configuration.

In [ ]:
import torch
import os

print("Environment Check")
print("=" * 50)

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Available: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("No GPU detected! Check NVIDIA drivers and CUDA installation.")

print(f"\nPyTorch Version: {torch.__version__}")
print(f"Working Directory: {os.getcwd()}")

In [ ]:
# Verify key packages are installed
import transformers
import peft
import bitsandbytes

print("Package Versions")
print("=" * 50)
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"torch: {torch.__version__}")
print("\nAll packages loaded successfully!")

In [ ]:
# Verify data files exist
print("Data Files")
print("=" * 50)

print("\nF5 Documentation:")
for f in os.listdir('data/f5_docs/'):
    print(f"  - {f}")

print("\nTraining Data:")
for f in os.listdir('data/training/'):
    print(f"  - {f}")

## 1.2 Load the Base Model

We'll use TinyLlama-1.1B-Chat with 4-bit quantization. This reduces the model size from ~4GB to ~2GB while maintaining reasonable quality.

### Why TinyLlama?
- **Small enough** for efficient training (~2GB VRAM with 4-bit)
- **Chat-optimized** for instruction following
- **Good baseline** to demonstrate enhancement techniques

In [ ]:
# =============================================================================
# WHY 4-BIT QUANTIZATION?
# =============================================================================
# Large language models require significant GPU memory. Quantization reduces
# the precision of model weights to use less memory:
#
#   Full precision (fp32):  4 bytes per parameter
#   Half precision (fp16):  2 bytes per parameter  
#   8-bit quantization:     1 byte per parameter
#   4-bit quantization:     0.5 bytes per parameter <- We use this!
#
# For TinyLlama (1.1B parameters):
#   - FP16: ~2.2 GB
#   - 4-bit: ~0.6 GB (fits easily on T4 with 16GB VRAM)
#
# BitsAndBytesConfig settings:
#   - load_in_4bit: Enable 4-bit quantization
#   - bnb_4bit_compute_dtype: Use fp16 for computations (faster on GPU)
#   - bnb_4bit_quant_type: "nf4" (Normal Float 4) is optimized for neural networks
# =============================================================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configure 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Enable 4-bit loading
    bnb_4bit_compute_dtype=torch.float16, # Use fp16 for matrix multiplications
    bnb_4bit_quant_type="nf4",            # NormalFloat4 - optimized for weights
    bnb_4bit_use_double_quant=False       # Single quantization (simpler)
)

print(f"Loading {MODEL_NAME}...")
print("This may take 1-2 minutes for the first download.")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded")
print(f"  Vocabulary size: {tokenizer.vocab_size:,}")

In [ ]:
# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"\nModel loaded successfully!")
print(f"  Model type: {type(model).__name__}")
print(f"  Device: {model.device}")

In [ ]:
# Check GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory Usage:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved: {reserved:.2f} GB")
    print(f"  Total: {total:.1f} GB")
    print(f"  Available: {total - reserved:.1f} GB")

## 1.3 Define the Generation Function

TinyLlama uses a specific chat template. We'll create a helper function for generating responses.

In [ ]:
# =============================================================================
# GENERATION PARAMETERS EXPLAINED
# =============================================================================
# How the model generates text is controlled by several parameters:
#
# max_new_tokens: Maximum length of generated response
#   - Higher = longer responses but slower and may ramble
#   - 256 is good for concise technical answers
#
# temperature: Controls randomness/creativity (0.0 to 2.0)
#   - 0.0 = deterministic (always same output)
#   - 0.7 = balanced creativity (good default)
#   - 1.0+ = more creative/random (may be less accurate)
#
# do_sample: Whether to use sampling vs greedy decoding
#   - True = sample from probability distribution (more varied)
#   - False = always pick highest probability token (repetitive)
#
# top_p (nucleus sampling): Only sample from top tokens that sum to p
#   - 0.9 = consider tokens comprising 90% of probability mass
#   - Helps avoid very unlikely/nonsensical tokens
# =============================================================================

def generate_response(question, max_new_tokens=256, temperature=0.7):
    """
    Generate a response from the model using TinyLlama's chat template.
    
    Args:
        question: The user's question
        max_new_tokens: Maximum tokens to generate (controls response length)
        temperature: Sampling temperature (higher = more creative/random)
    
    Returns:
        The model's response as a string
    """
    # Format the prompt using TinyLlama's expected chat structure
    prompt = f"""<|system|>
You are an F5 Networks technical expert. Provide accurate, detailed answers about BIG-IP, iRules, load balancing, SSL offloading, and other F5 technologies.</s>
<|user|>
{question}</s>
<|assistant|>
"""
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode tokens back to text
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the assistant's response
    if "<|assistant|>" in full_response:
        response = full_response.split("<|assistant|>")[-1].strip()
    else:
        response = full_response[len(prompt):].strip()
    
    return response

print("Generation function defined")

## 1.4 Test Basic Generation

Let's make sure the model is working with a simple test.

In [ ]:
# Simple test
test_question = "What is Python?"
print(f"Question: {test_question}\n")
print(f"Response: {generate_response(test_question, max_new_tokens=100)}")

## 1.5 Baseline Evaluation on F5 Questions

Now let's test the model on F5-specific questions to establish our baseline.

In [ ]:
# F5 Technical Questions for Baseline Evaluation
eval_questions = [
    "What is a virtual server in F5 BIG-IP and what is its purpose?",
    "How do I configure SSL offloading on F5 BIG-IP?",
    "Write an iRule to redirect HTTP to HTTPS.",
    "What is the difference between Least Connections and Round Robin load balancing?",
    "How do I troubleshoot a pool member that is marked as offline?"
]

print(f"Evaluating {len(eval_questions)} questions...\n")
print("=" * 80)

In [ ]:
# Store baseline responses
baseline_responses = []

for i, question in enumerate(eval_questions, 1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {question}")
    print("-" * 80)
    
    response = generate_response(question)
    baseline_responses.append({
        "question": question,
        "response": response
    })
    
    print(f"Response:\n{response}")

print(f"\n{'='*80}")
print("Baseline evaluation complete!")

## 1.6 Analysis: Baseline Limitations

Review the responses above. Consider:

1. **Accuracy**: Are the technical details correct?
2. **Specificity**: Does it mention F5-specific terminology and features?
3. **Completeness**: Does it fully answer the question?
4. **Actionability**: Could someone follow this advice?

You may notice the baseline model:
- Provides generic networking answers
- Lacks F5-specific terminology (TMSH, iRules, profiles)
- May make up plausible-sounding but incorrect information
- Doesn't reference actual configuration commands

## 1.7 Save Baseline Results

Let's save these results for comparison in Module 4.

In [ ]:
import json

# Create results directory
os.makedirs("results", exist_ok=True)

# Save baseline results
results = {
    "model": MODEL_NAME,
    "type": "baseline",
    "responses": baseline_responses
}

with open("results/baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Baseline results saved to results/baseline_results.json")

## 1.8 Summary

In this module, we:

1. Verified GPU environment configuration
2. Loaded TinyLlama-1.1B with 4-bit quantization
3. Tested baseline performance on F5 questions
4. Identified limitations in domain-specific knowledge

### Key Observations
- The base model has general knowledge but lacks F5-specific expertise
- 4-bit quantization allows running a 1.1B model efficiently
- We have a clear baseline to measure improvements against

### Next Steps
In **Module 2**, we'll build a RAG system that augments the model with F5 documentation.

In [ ]:
# Cleanup - free GPU memory before the next module
import gc
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared - ready for Module 2")